# 06 · The answer, the board number, and the cold run

**Produces:** `slides/board_number.xlsx` (the one number with its denominator, period and comparison beside it), the headline printed in the last cell, and a cold-run row in the decisions log.

**Needs:** notebooks 03 and 05 run.

This notebook is short on purpose. It is the one the panel will ask you to run from a fresh Codespace, so it reads clean tables and does nothing clever.

## CONFIG

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "4",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

3.0.6 2.5.4


In [2]:
bk = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
h = {"last": bk[bk["slot_at"].between(*CONFIG["period_last"])], "this": bk[bk["slot_at"].between(*CONFIG["period_this"])]}
tests_last, tests_this = int(h["last"]["n_tests"].sum()), int(h["this"]["n_tests"].sum())
growth = (tests_this / tests_last - 1) * 100
HEADLINE = {"claim": "<one sentence for the board>", "number": round(growth, 1), "numerator": tests_this, "denominator": tests_last,
            "unit": "tests on bookings that were collected or no-show, after the identity rule in docs/decisions-log.md",
            "period": "H1 2026 against H1 2025, six cities", "comparison": "plan: +18 percent", "caveat": "<the one thing that would change the claim>"}
for k, v in HEADLINE.items(): print(f"{k:>12}: {v}")

       claim: <one sentence for the board>
      number: 5.0
   numerator: 452933
 denominator: 431261
        unit: tests on bookings that were collected or no-show, after the identity rule in docs/decisions-log.md
      period: H1 2026 against H1 2025, six cities
  comparison: plan: +18 percent
      caveat: <the one thing that would change the claim>


## The board number, in Excel, with its sentence beside it

In [3]:
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
wb = Workbook(); ws = wb.active; ws.title = "Board number"
rows = [("Kalpa Health · Brief " + CONFIG["brief"] + " · Group " + CONFIG["group"], ""), ("", ""),
        ("The number", f"{HEADLINE['number']:+.1f}%"), ("What it counts", HEADLINE["unit"]), ("Numerator", HEADLINE["numerator"]), ("Denominator", HEADLINE["denominator"]),
        ("Period", HEADLINE["period"]), ("Compared with", HEADLINE["comparison"]), ("The sentence", HEADLINE["claim"]), ("The caveat", HEADLINE["caveat"]),
        ("Formula check", "=ROUND((B5/B6-1)*100,1)")]
for r in rows: ws.append(list(r))
ws["A1"].font = Font(bold=True, size=14); ws["B3"].font = Font(bold=True, size=24)
ws.column_dimensions["A"].width = 18; ws.column_dimensions["B"].width = 90
for row in ws.iter_rows(min_row=3, max_row=11): row[1].alignment = Alignment(wrap_text=True, vertical="top")
(ROOT / "slides").mkdir(exist_ok=True)
wb.save(ROOT / "slides" / "board_number.xlsx")
print("saved slides/board_number.xlsx; the formula in B11 must agree with B3 when opened")

saved slides/board_number.xlsx; the formula in B11 must agree with B3 when opened


## The cold run

In [4]:
import subprocess, sys
try:
    sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
except Exception:
    sha = "no git"
print(f"commit {sha}  headline {HEADLINE['number']:+.1f}%  ({HEADLINE['numerator']:,} / {HEADLINE['denominator']:,})")
# Record it the moment it is run. Uncomment and fill in.
# logs.cold_run(who="<name>", fresh=True, runtime="<m:ss>", headline=f"{HEADLINE['number']:+.1f}%", same_as_last="yes", why="")
checks.check("headline is a finite number", np.isfinite(HEADLINE["number"]))
checks.check("the claim sentence has been written", not HEADLINE["claim"].startswith("<"))
checks.summary()

commit e20ba44  headline +5.0%  (452,933 / 431,261)
PASS  headline is a finite number
FAIL  the claim sentence has been written
1 of 2 checks passed


,check,passed,detail
0,headline is a finite number,True,
1,the claim sentence has been written,False,


## The last cell

The number on slide 3 is the one printed above. If it is not, one of the two is wrong, and the panel will ask which.